# AB тестирование часть 4

## Unit of Randomization (Единица рандомизации)

Определение:
**Единица рандомизации** — это объект, на уровне которого происходит случайное распределение в группы A/B / минимальная сущность, которую ты можешь “изолировать” от влияния treatment

Варианты:

- Пользователь (user-level)
- Сессия (session-level)
- Устройство (device-level)
- Кластер (например, город, компания, команда)


Trade-off (это ключевой слайд)

- Чем мельче единица → тем больше данных → выше мощность данных
- Чем крупнее единица → тем меньше смещение → но ниже мощность данных

Это фундаментальный компромисс:

session → много данных, но зависимость
user → меньше данных, но независимость
cluster → ещё меньше данных, но честный causal effect

> Это подтверждается практикой: при кластерной рандомизации растёт дисперсия и падает мощность из-за меньшего числа независимых единиц

### Алгоритм выбора единицы рандомизации

#### Что ты измеряешь?

| Метрика            | Правильная единица      |
| ------------------ | ----------------------- |
| Revenue per user   | user                    |
| CTR | просмотр / session    |
| Retention (процент возврата клиента)          | user                    |
| Общий объем продаж   | user / seller / cluster |


#### Есть ли зависимость между наблюдениями?

Задай себе вопросы:

- Один пользователь может попасть в обе группы?
- Пользователи влияют друг на друга?
- Есть влияющие сторонние метрики (например, рекомендации, цены)?

Так, например:
1) Пользователь может сделать разные 10 сессий и попасть в каждую из них. Следовательно, он может попасть и в группу A, и в группу B, следовательно, нужно менять единицу рандомизации;
2) Каждый пользователь попадает в одну группу, следовательно, мы можем гарантировать наличие пользователя либо в одной, либо в другой группе;
3) Один пользователь может повлиять на действия другого пользователя. Следовательно, мы не можем гарантировать, что изменения действия пользователей связаны только с нашими изменениями.

## Cluster-level

Если с пользователями и сессиями понятно, то можно ли изолировать 3 пример? Можно, с использованием кластеризации.

Пример:

- соцсеть
- маркетплейс
- multiplayer игра

Например так происходит AB тестирование в google docs
https://www.unofficialgoogledatascience.com/2018/01/designing-ab-tests-in-collaboration.html

## Алгоритм выбора рандомизации:
Определи метрику
- Проверь независимость
- Минимизируй unit
- Проверь interference

Trade-off:

1) ↓ unit → ↑ power, ↑ bias
2) ↑ unit → ↓ bias, ↓ power

# Защитные метрики (Guardrail metrics)

Guardrail metrics — это метрики, которые не должны ухудшиться в результате эксперимента.

Если основная метрика отвечает на вопрос:

- стало ли лучше?

то guardrails отвечают:

- не сломали ли мы что-то важное?

Ты тестируешь новый дизайн кнопки.

Результат:

CTR +10%

Кажется, успех.

Но смотри guardrails:

bounce rate (процент отказов) ↑
retention (повторные действия) ↓

Вывод:

👉 ты сделал интерфейс более “кликабельным”, но хуже по удержанию

### Типы guardrail метрик

Обычно их делят на 3 категории:

#### 1. Product health metrics

Отражают общее состояние продукта.

Примеры:

1) retention (повторные действия)
2) DAU / WAU (количество пользователей за сутки)
3) session length (длина сессии)

#### 2. UX / quality metrics

Показывают качество взаимодействия.

1) bounce rate (процент отказов)
2) time to load (latency) (задержка)
3) error rate (количество ошибок)


#### Business metrics

Связаны с деньгами.

- revenue (выручка)
- ARPU (выручка на пользователя)
- conversion в оплату

### Как выбрать?

Определить риски изменения

Что может сломаться?

| изменение             | риск             |
| --------------------- | ---------------- |
| новый UI              | ухудшение UX     |
| алгоритм рекомендаций | перекос контента |
| оптимизация скорости  | потеря качества  |


**Под каждый риск — метрика**

Пример: ухудшение UX → guardrail: bounce rate

Определение защитной метрики такое же, как и в AB: определяем ее изменение и проверям

# Влияние поведения пользователей друг на друга (Interference / Network Effects)

Иногда в результате A/B тестирования на платформах, где разные пользователи взаимодействуют друг с другом можно наблюдать эффекты этого влияния. К таким платформам можно отнести:
- Социальные сети (лайки, рекомендации)
- Маркетплейсы (продавцы/покупатели)
- Игры (мультиплеер)
- Рекомендательные системы

В результате нарушается ключевое допущение A/B теста: независимость наблюдений (SUTVA)

## Варианты работы с interference

### Cluster Randomization

- Рандомизация групп пользователей (например, по регионам)
- Снижает cross-group влияние

Мы можем производит рандомизацию внутри разных кластеров пользователей. Таким образом это снижает влияние поведения пользователей (применяется в ситуациях, когда поведение локально у пользователей)

### Graph-based подходы

- Учет социальных связей
- Разделение графа на кластеры

В основном применяется в социальных сетях где имеется информация о связях пользователей друг с другом. AB тестирование не должно в данном случае содержать пересечений данного графа для того, чтобы пользователи не влияли друг на друга (по интересам, по предпочтениям и т. д.). Внутри графов может производится рандомизация

### Switchback Experiments
- Рандомизация по времени (например, часам)
- Подходит для систем с высокой нагрузкой и высоконгрузочных метрик

### Two-sided experiments

- Отдельное управление treatment для разных сторон (например, buyer/seller)

# Эффект "новизны" (Novelty Effect)

Это эффект, который определяет временное поведение пользователей исходя из новизны дизайна, а не из-за его удобства или других характеристик.

Как проявляется:

- Резкий рост метрик сразу после запуска
- Последующее снижение
- Иногда — наоборот (пользователи сначала путаются)

Следовательно, если эксперемент не продлиться необходимое количетсво времени до снижения этого эффекта получится, что мы можем задетектировать эффекты, которых на самом деле не существует. Может отражаться как снижением метрики, так и ее завышением

Как работать:

1) Удлинение эксперимента
    - Минимум 2–4 недели (зависит от продукта)
2) Анализ по времени
    - Разбивка эффекта по дням/неделям
    - Проверка стабилизации
3) Ramp-up стратегия
    - Постепенное включение пользователей

# Ramp-up стратегия (постеменное увеличение доли в B группе)

Ramp-up — это постепенное увеличение доли пользователей, которым показывается новая версия фичи или treatment.

Например:

1% → 5% → 10% → 25% → 50% → 100%

На каждом шаге проверяются:

- защитные метрики
- системные метрики
- ошибки
- бизнес-метрики

Если появляется проблема — rollout останавливают или откатывают.

### Причины использования rump-up

#### 1) Управление риском (самая главная причина)

Когда ты выкатываешь изменения сразу на 100% пользователей — ты ставишь на кон весь продукт.

Ramp-up позволяет:

- обнаружить баги
- обнаружить падение метрик
- быстро откатить изменения

#### 2) Раннее обнаружение проблем

Очень многие проблемы проявляются только в production.

Например:

- неожиданные баги
- нагрузка на backend
- проблемы с кэшами
- деградация latency

Ramp-up помогает обнаружить их до того, как они ударят по всей системе.

#### 3. Баланс между скоростью и безопасностью

Компании постоянно балансируют: скорость разработки  vs  риск поломки

Ramp-up — это компромисс:

- если всё хорошо → rollout ускоряется
- если появляются проблемы → rollout останавливается

Некоторые компании даже используют алгоритмы, которые автоматически определяют когда увеличивать процент пользователей.

### Где используется ramp-up (очень важный слайд)

#### A/B тесты

Самый очевидный случай.

Пример:

- Day 1: 1%
- Day 2: 5%
- Day 3: 10%
- Day 5: 50%

Если метрики норм — продолжаем.

#### Критические продуктовые изменения

Например:

- новый checkout
- новый pricing
- новый алгоритм рекомендаций
- redesign всего интерфейса

Если сделать это сразу на 50% пользователей — можно сломать revenue.

#### Инфраструктурные изменения

Очень частый кейс:

- новый ML-модель
- новый ranking алгоритм
- миграция на новый сервис
- новый database layer

Ramp-up позволяет:

10% трафика → новая система
90% → старая

#### Performance изменения

Например:

- новый caching layer
- новый CDN
- новый search engine

Если задержка увеличится — это будет видно сразу.

Ramp-up может улучшить статистическую оценку эффекта.

Потому что у тебя появляется несколько экспериментов с разными долями treatment.

https://arxiv.org/abs/2410.12740

# Сезонность

Сезонность — систематические изменения метрик во времени (дни недели, месяцы, праздники и т.д.)

Например:
- Выходные vs будни
- Черная пятница / праздники
- Зарплатные дни
- Летний/зимний сезон

Проблемы:
- Искажение эффекта
- Ложные позитивы/негативы
- Рост дисперсии

### Учет сезонности
1. Длительность эксперимента
- Минимум: полный недельный цикл
- Лучше: несколько циклов
2. Стратификация
Балансировка по:
- дням недели
- регионам
- типам пользователей
3. Difference-in-Differences
Сравнение изменений, а не уровней
4. Time-series модели
Учет трендов и сезонных компонент
Например:
- ARIMA / SARIMA
- Bayesian structural time series
5. CUPED / ковариаты
- Использование исторических данных для снижения шума
6. Switchback дизайн
- Рандомизация по времени помогает нивелировать сезонность